# Get Grid of California Addresses and their transportation info to campus, as well as gas prices and electric vehicle charging rates, using Python and relevant APIs

---

## Import Libraries, Set Up API Keys

In [18]:
import numpy as np 
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
import googlemaps

load_dotenv('.env')

gmaps = googlemaps.Client(key=os.getenv('API_KEY'))

## Within range of lat/lng pairs, reverse geocode lat/lng pairs to valid addresses, put into a Dataframe, then gather getDirections data from each valid address and put into column in Dataframe across multiple transportation modes

In [19]:
grid_df = pd.DataFrame(columns = ['latitude', 'longitude', 'address', 'driving_time_to_campus', 'walking_time_to_campus', 'bicycling_time_to_campus', 'transit_time_to_campus'])
latitudes = np.arange(40.0, 41.3, 0.01)  # Northern latitude range for CA county
longitudes = np.arange(-124.5, -123.0, 0.01)  # Western longitude range for CA county

In [20]:
for lat in latitudes:
    for lng in longitudes:
        # Reverse geocode to get address
        reverse_geocode_result = gmaps.reverse_geocode((lat, lng))
        if reverse_geocode_result:
            address = reverse_geocode_result[0]['formatted_address']
            
            # Get directions to campus for different modes of transportation
            # Address: 1 Harpst St. Arcata CA 95521
            driving_directions = gmaps.directions(address, "California State Polytechnic University, Humboldt", mode="driving")
            walking_directions = gmaps.directions(address, "California State Polytechnic University, Humboldt", mode="walking")
            bicycling_directions = gmaps.directions(address, "California State Polytechnic University, Humboldt", mode="bicycling")
            transit_directions = gmaps.directions(address, "California State Polytechnic University, Humboldt", mode="transit")
            if driving_directions and walking_directions and bicycling_directions and transit_directions:
                driving_time = driving_directions[0]['legs'][0]['duration']['text']
                walking_time = walking_directions[0]['legs'][0]['duration']['text']
                bicycling_time = bicycling_directions[0]['legs'][0]['duration']['text']
                transit_time = transit_directions[0]['legs'][0]['duration']['text']
                
                # Append to DataFrame
                grid_df = pd.concat([grid_df, pd.DataFrame({
                    'latitude': [lat],
                    'longitude': [lng],
                    'address': [address],
                    'driving_time_to_campus': [driving_time],
                    'walking_time_to_campus': [walking_time],
                    'bicycling_time_to_campus': [bicycling_time],
                    'transit_time_to_campus': [transit_time]
                })], ignore_index=True)

C:\Users\wolfe\AppData\Local\Temp\ipykernel_31464\1577985607.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grid_df = pd.concat([grid_df, pd.DataFrame({


In [21]:
grid_df.shape

(775, 7)

In [23]:
grid_df.head()

,latitude,longitude,address,driving_time_to_campus,walking_time_to_campus,bicycling_time_to_campus,transit_time_to_campus
0,40.06,-123.80,"356X+2X Benbow, CA, USA",1 hour 27 mins,1 day 13 hours,11 hours 18 mins,3 hours 26 mins
1,40.06,-123.79,"3666+22 Benbow, CA, USA",1 hour 31 mins,1 day 13 hours,11 hours 19 mins,3 hours 24 mins
2,40.06,-123.78,"3669+2X Benbow, CA, USA",1 hour 22 mins,1 day 13 hours,11 hours 15 mins,3 hours 10 mins
3,40.06,-123.77,"366J+22 Benbow, CA, USA",1 hour 27 mins,1 day 13 hours,11 hours 20 mins,3 hours 30 mins
4,40.07,-123.81,"359R+X2 Garberville, CA, USA",1 hour 27 mins,1 day 13 hours,11 hours 20 mins,9 hours 41 mins


In [22]:
grid_df.to_csv('../clean_data/Grid_Data_01.csv', index=False)

## Convert Text of Transportation times into hours in Dataframe, then save Dataframe as CSV again

## Plot Histograms of Transportation times across different modes of transportation to identify any modality that allows us to cluster addresses by transportation time to campus

## Create new column for each transport method that labels them by clusters of transportation time to campus

## Gather Gas and Electric Vehicle Charging Prices for each address using relevant APIs, put into Dataframe

# Done